In [1]:
# Descarga automática de taxi_zone_lookup.csv si no existe
import os
import requests
zone_lookup_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_lookup_path = "/home/jovyan/data/nyc_taxi_parquet/taxi_zone_lookup.csv"
os.makedirs(os.path.dirname(zone_lookup_path), exist_ok=True)
if not os.path.exists(zone_lookup_path):
    print("Descargando taxi_zone_lookup.csv ...")
    try:
        r = requests.get(zone_lookup_url, stream=True)
        if r.status_code == 200:
            with open(zone_lookup_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Descargado: taxi_zone_lookup.csv")
        else:
            print("Archivo taxi_zone_lookup.csv no encontrado en el servidor.")
    except Exception as e:
        print(f"Error descargando taxi_zone_lookup.csv: {e}")
else:
    print("taxi_zone_lookup.csv ya existe.")

taxi_zone_lookup.csv ya existe.


In [1]:
# Demostración variables entornos de .env
import os
print(os.getenv("PG_HOST"), os.getenv("PG_DB"), os.getenv("PG_USER"))

postgres nyc_taxi postgres


In [2]:
# Crear esquema RAW si no existe antes de cualquier ingesta o creación de tablas
import psycopg2
import os

conn = psycopg2.connect(host=os.getenv('PG_HOST'), port=os.getenv('PG_PORT'), dbname=os.getenv('PG_DB'), user=os.getenv('PG_USER'), password=os.getenv('PG_PASSWORD'))
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")
conn.commit()
cur.close()
conn.close()
print('Esquema raw listo.')

Esquema raw listo.


In [3]:
import os
import requests

# Descarga bajo demanda: helper para traer un Parquet solo cuando se necesita
# y evitar acumulación de archivos locales.
def download_to(path: str, url: str, chunk_size: int = 8192) -> str:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    r = requests.get(url, stream=True, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"No encontrado en origen: {url} (HTTP {r.status_code})")
    with open(path, "wb") as f:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
    return path

print("Modo descarga bajo demanda listo (sin staging masivo).")

Modo descarga bajo demanda listo (sin staging masivo).


## Descarga automática de archivos Parquet NYC TLC
Descarga los archivos Parquet necesarios directamente en el contenedor para evitar problemas de rutas y asegurar reproducibilidad.

# 01 - Ingesta Parquet NYC TLC (2015–2025)
Este notebook realiza la ingesta masiva de los archivos Parquet (Yellow/Green) del dataset NYC TLC hacia Postgres en el esquema raw.

## 1. Configuración de entorno
Carga de variables de entorno y librerías necesarias.

In [4]:
import os
from pyspark.sql import SparkSession
import psycopg2
from datetime import datetime

## 2. Inicializar Spark Session

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC TLC Ingesta Parquet") \
    .getOrCreate()

In [11]:
# (Opcional) Crear índices para acelerar verificación/consultas y ANALYZE con progreso en vivo
import os
import time
import threading
import psycopg2

PG_HOST = os.getenv('PG_HOST')
PG_PORT = os.getenv('PG_PORT')
PG_DB = os.getenv('PG_DB')
PG_USER = os.getenv('PG_USER')
PG_PASSWORD = os.getenv('PG_PASSWORD')
# Crea solo los índices por (year, month) por defecto. Activa los de 'ingested_at_utc' poniendo CREATE_INGESTED_IDX=1
CREATE_INGESTED_IDX = os.getenv('CREATE_INGESTED_IDX', '0') == '1'


def _poll_index_progress(pid: int, stop_event: threading.Event):
    """Imprime el progreso de CREATE INDEX usando vistas de pg_stat cada ~5s."""
    while not stop_event.is_set():
        try:
            conn2 = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD, connect_timeout=5)
            conn2.autocommit = True
            with conn2.cursor() as cur2:
                # Intenta leer progreso detallado (PG >= 12)
                try:
                    cur2.execute(
                        """
                        SELECT phase,
                               relid::regclass::text AS table_name,
                               COALESCE(tuples_done,0), COALESCE(tuples_total,0),
                               COALESCE(blocks_done,0), COALESCE(blocks_total,0)
                        FROM pg_stat_progress_create_index
                        WHERE pid = %s
                        """,
                        (pid,),
                    )
                    rows = cur2.fetchall()
                    if rows:
                        phase, table_name, t_done, t_tot, b_done, b_tot = rows[0]
                        pct_t = f"{(t_done / t_tot * 100):.1f}%" if t_tot else "?"
                        pct_b = f"{(b_done / b_tot * 100):.1f}%" if b_tot else "?"
                        print(f"  [idx pid={pid}] {table_name} | phase={phase} | tuples={t_done}/{t_tot} ({pct_t}) | blocks={b_done}/{b_tot} ({pct_b})", flush=True)
                    else:
                        # Fallback: al menos reporta el runtime
                        cur2.execute("SELECT now() - query_start FROM pg_stat_activity WHERE pid = %s", (pid,))
                        r = cur2.fetchone()
                        if r:
                            print(f"  [idx pid={pid}] ejecutándose hace {r[0]}", flush=True)
                except Exception:
                    # Fallback simple si la vista/columnas no existen
                    cur2.execute("SELECT now() - query_start FROM pg_stat_activity WHERE pid = %s", (pid,))
                    r = cur2.fetchone()
                    if r:
                        print(f"  [idx pid={pid}] ejecutándose hace {r[0]}", flush=True)
        except Exception:
            pass
        finally:
            try:
                conn2.close()
            except Exception:
                pass
        stop_event.wait(5)


try:
    conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD)
    conn.autocommit = True  # requerido para CREATE INDEX CONCURRENTLY
    cur = conn.cursor()

    part_idx = [
        "CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_yellow_source_ym ON raw.yellow_taxi_trip (source_year, source_month)",
        "CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_green_source_ym  ON raw.green_taxi_trip  (source_year, source_month)",
    ]
    ingested_idx = [
        "CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_yellow_ingested  ON raw.yellow_taxi_trip (ingested_at_utc)",
        "CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_green_ingested   ON raw.green_taxi_trip  (ingested_at_utc)",
    ]

    stmts = part_idx + (ingested_idx if CREATE_INGESTED_IDX else [])

    print(f"Creación de índices (CREATE_INGESTED_IDX={int(CREATE_INGESTED_IDX)})", flush=True)

    for i, s in enumerate(stmts, start=1):
        try:
            print(f"[{i}/{len(stmts)}] Iniciando: {s.split(' ON ')[0]} ...", flush=True)
            # Obtiene el PID del backend de esta sesión para monitoreo
            cur.execute("SELECT pg_backend_pid();")
            pid = cur.fetchone()[0]
            stop_event = threading.Event()
            t = threading.Thread(target=_poll_index_progress, args=(pid, stop_event), daemon=True)
            t.start()
            t0 = time.perf_counter()
            cur.execute(s)  # Bloquea hasta terminar la creación del índice
            dt = time.perf_counter() - t0
            stop_event.set(); t.join(timeout=1)
            print(f"[{i}/{len(stmts)}] OK en {dt:,.1f}s\n", flush=True)
        except Exception as e:
            try:
                stop_event.set(); t.join(timeout=1)
            except Exception:
                pass
            print("Aviso (índice):", e, flush=True)

    # Actualiza estadísticas por tabla (más rápido que CREATE INDEX)
    for tname in ["raw.yellow_taxi_trip", "raw.green_taxi_trip"]:
        try:
            print(f"ANALYZE {tname} ...", flush=True)
            t0 = time.perf_counter()
            cur.execute(f"ANALYZE {tname};")
            dt = time.perf_counter() - t0
            print(f"ANALYZE {tname} OK en {dt:,.1f}s", flush=True)
        except Exception as e:
            print("Aviso (ANALYZE):", e, flush=True)

    cur.close()
    conn.close()
    print("Índices/ANALYZE listos (opcional).", flush=True)
except Exception as e:
    print("No se pudieron crear índices (opcional):", e, flush=True)

Creación de índices (CREATE_INGESTED_IDX=0)
[1/2] Iniciando: CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_yellow_source_ym ...
  [idx pid=1818] raw.yellow_taxi_trip | phase=waiting for writers before build | tuples=0/0 (?) | blocks=0/0 (?)
  [idx pid=1818] raw.yellow_taxi_trip | phase=building index: scanning table | tuples=0/0 (?) | blocks=19296/12907860 (0.1%)
  [idx pid=1818] raw.yellow_taxi_trip | phase=building index: scanning table | tuples=0/0 (?) | blocks=39022/12907860 (0.3%)
  [idx pid=1818] raw.yellow_taxi_trip | phase=building index: scanning table | tuples=0/0 (?) | blocks=66571/12907860 (0.5%)
  [idx pid=1818] raw.yellow_taxi_trip | phase=building index: scanning table | tuples=0/0 (?) | blocks=84365/12907860 (0.7%)
  [idx pid=1818] raw.yellow_taxi_trip | phase=building index: scanning table | tuples=0/0 (?) | blocks=102473/12907860 (0.8%)
  [idx pid=1818] raw.yellow_taxi_trip | phase=building index: scanning table | tuples=0/0 (?) | blocks=118883/12907860 (0.9%)
  [idx pi

## Optimización de verificación y escritura (acelera la ingesta)

Para evitar que la verificación se vuelva lenta a medida que crecen las tablas:
- Se cambia `COUNT(*)` por `EXISTS` (mucho más rápido) y se fija un `statement_timeout` corto.
- Se habilita escritura JDBC con inserciones batched (`rewriteBatchedInserts=true`).
- Variables opcionales:
  - `CHECK_EXISTS` ("1" por defecto): permite desactivar la verificación de existencia por partición si quieres maximizar velocidad.
  - `COUNT_ROWS` ("0" por defecto): activa/desactiva el `df.count()` por lote (puedes activarlo al final para métricas definitivas).
- Índices opcionales en `(source_year, source_month)` e `ingested_at_utc` para consultas rápidas por partición o fecha de ingesta.

## 3. Prueba de ingesta
Solo se ingesta un mes para validar el proceso y evitar uso innecesario de disco local.

In [11]:
# Ingesta YELLOW (2025) con alineación de esquema a Postgres
# - Descarga bajo demanda SOLO el mes necesario
# - Escribe en Postgres en modo append
# - Alinea columnas al esquema real de raw.yellow_taxi_trip: agrega faltantes como NULL y descarta extras (p.ej., cbd_congestion_fee)
# - Optimizado: EXISTS en vez de COUNT(*), timeout corto y JDBC batched inserts
import os
import psycopg2
from typing import List, Tuple
from pyspark.sql.functions import lit
from pyspark.sql.types import (
    StringType, IntegerType, LongType, DoubleType, BooleanType, TimestampType
)

# Flags de control
CHECK_EXISTS = os.getenv('CHECK_EXISTS', '1') == '1'
COUNT_ROWS  = os.getenv('COUNT_ROWS',  '0') == '1'

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/"
local_data_path = "/home/jovyan/data/nyc_taxi_parquet"
os.makedirs(local_data_path, exist_ok=True)

# JDBC URL con inserciones batched (acelera escrituras masivas)
jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}?rewriteBatchedInserts=true"

print(f"Inicio de ingesta condicional (sin staging masivo). CHECK_EXISTS={int(CHECK_EXISTS)} COUNT_ROWS={int(COUNT_ROWS)}", flush=True)

# Helpers mínimos para esquema destino

def _connect():
    return psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD, connect_timeout=10)

_pg_to_spark = {
    'integer': IntegerType(),
    'bigint': LongType(),
    'smallint': IntegerType(),
    'double precision': DoubleType(),
    'real': DoubleType(),
    'numeric': DoubleType(),
    'decimal': DoubleType(),
    'boolean': BooleanType(),
    'text': StringType(),
    'character varying': StringType(),
    'varchar': StringType(),
    'timestamp without time zone': TimestampType(),
    'timestamp with time zone': TimestampType(),
    'timestamp': TimestampType(),
}

def get_table_schema(schema: str, table: str) -> List[Tuple[str, str]]:
    with _connect() as conn, conn.cursor() as cur:
        cur.execute(
            """
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = %s AND table_name = %s
            ORDER BY ordinal_position
            """,
            (schema, table),
        )
        return cur.fetchall()


def unify_df_to_table(df, table_cols: List[Tuple[str, str]]):
    # Renombrar todas las columnas a minúsculas para coincidir con nombres no comillados en PG
    for c in df.columns:
        lc = c.lower()
        if lc != c:
            df = df.withColumnRenamed(c, lc)
    df_cols = set(df.columns)

    # Asegurar existencia y tipo para cada columna de la tabla
    from pyspark.sql.functions import lit as _lit
    for col_name, pg_type in table_cols:
        if col_name not in df_cols:
            spark_type = _pg_to_spark.get(pg_type, StringType())
            df = df.withColumn(col_name, _lit(None).cast(spark_type))
    # Seleccionar únicamente columnas de destino en orden
    ordered_cols = [name for name, _ in table_cols]
    return df.select(*ordered_cols)

# Leer esquema actual de la tabla destino si existe
try:
    target_cols = get_table_schema(PG_SCHEMA_RAW, 'yellow_taxi_trip')
    if not target_cols:
        print("Aviso: raw.yellow_taxi_trip no existe aún; se creará con el primer append.")
        target_cols = None
    else:
        print(f"Tabla destino encontrada con {len(target_cols)} columnas.")
except Exception as e:
    print("No se pudo leer el esquema de la tabla destino (se intentará crear automáticamente):", e)
    target_cols = None

for service in ['yellow']:
    for year in range(2025, 2026):
        for month in range(3, 4):
            file_name = f'{service}_tripdata_{year}-{month:02d}.parquet'
            url = f'{base_url}{file_name}'
            parquet_file = os.path.join(local_data_path, file_name)

            # Progreso inmediato
            print(f"[{service} {year}-{month:02d}] Preparando ingesta...", flush=True)

            # Verificar existencia en Postgres de forma ligera (opcional)
            already_loaded = False
            if CHECK_EXISTS:
                try:
                    with _connect() as conn, conn.cursor() as cur:
                        try:
                            cur.execute("SET LOCAL statement_timeout = '5s';")
                        except Exception:
                            pass
                        cur.execute(
                            f"""
                            SELECT EXISTS (
                                SELECT 1
                                FROM {PG_SCHEMA_RAW}.{service}_taxi_trip
                                WHERE source_year = %s AND source_month = %s
                            );
                            """,
                            (year, month),
                        )
                        already_loaded = bool(cur.fetchone()[0])
                except Exception as e:
                    print(f"[{service} {year}-{month:02d}] Aviso: no se pudo consultar Postgres (se continúa): {e}", flush=True)

            if already_loaded:
                if os.path.exists(parquet_file):
                    try:
                        os.remove(parquet_file)
                        print(f"[{service} {year}-{month:02d}] Archivo local removido (ya cargado en Postgres): {file_name}", flush=True)
                    except Exception as e:
                        print(f"[{service} {year}-{month:02d}] No se pudo eliminar {file_name}: {e}", flush=True)
                print(f"[{service} {year}-{month:02d}] Ya existe en Postgres. Se omite la ingesta.", flush=True)
                continue

            # Descarga bajo demanda (si no existe localmente)
            if not os.path.exists(parquet_file):
                try:
                    print(f"[{service} {year}-{month:02d}] Descargando {file_name} ...", flush=True)
                    download_to(parquet_file, url)
                    print(f"[{service} {year}-{month:02d}] Descargado: {file_name}", flush=True)
                except Exception as e:
                    print(f"[{service} {year}-{month:02d}] Archivo no disponible {file_name}: {e}", flush=True)
                    continue

            # Leer Parquet, normalizar, escribir, limpiar
            try:
                print(f"[{service} {year}-{month:02d}] Leyendo parquet...", flush=True)
                df = spark.read.parquet(parquet_file)

                # Metadatos
                df = (df
                      .withColumn('service_type', lit(service))
                      .withColumn('source_year', lit(year))
                      .withColumn('source_month', lit(month))
                      .withColumn('ingested_at_utc', lit(datetime.utcnow()))
                )

                # Alinear al esquema de la tabla destino si lo conocemos; si no, al menos pasar todo a lowercase
                if target_cols is not None:
                    df = unify_df_to_table(df, target_cols)
                else:
                    for c in df.columns:
                        lc = c.lower()
                        if c != lc:
                            df = df.withColumnRenamed(c, lc)

                # (Opcional) Conteo exacto
                if COUNT_ROWS:
                    n_rows = df.count(); print(f'Filas leídas: {n_rows}', flush=True)
                else:
                    n_rows = None; print('Filas leídas: N/A (COUNT_ROWS=0)', flush=True)

                # Escribir en Postgres (batched inserts)
                print(f"[{service} {year}-{month:02d}] Escribiendo en Postgres...", flush=True)
                (
                    df.write
                      .format('jdbc')
                      .option('url', jdbc_url)
                      .option('dbtable', f'{PG_SCHEMA_RAW}.{service}_taxi_trip')
                      .option('user', PG_USER)
                      .option('password', PG_PASSWORD)
                      .option('batchsize', '10000')
                      .mode('append')
                      .save()
                )
                if n_rows is not None:
                    print(f'Ingesta exitosa: {service} {year}-{month:02d} | Filas: {n_rows}', flush=True)
                else:
                    print(f'Ingesta exitosa: {service} {year}-{month:02d}', flush=True)
            except Exception as e:
                print(f'[{service} {year}-{month:02d}] Error en ingesta: {e}', flush=True)
            finally:
                if os.path.exists(parquet_file):
                    try:
                        os.remove(parquet_file)
                        print(f"[{service} {year}-{month:02d}] Eliminado local: {file_name}", flush=True)
                    except Exception as e:
                        print(f"[{service} {year}-{month:02d}] No se pudo eliminar {file_name}: {e}", flush=True)

Inicio de ingesta condicional (sin staging masivo). CHECK_EXISTS=1 COUNT_ROWS=0
Tabla destino encontrada con 23 columnas.
[yellow 2025-03] Preparando ingesta...
[yellow 2025-03] Descargando yellow_tripdata_2025-03.parquet ...
[yellow 2025-03] Descargado: yellow_tripdata_2025-03.parquet
[yellow 2025-03] Leyendo parquet...
Filas leídas: N/A (COUNT_ROWS=0)
[yellow 2025-03] Escribiendo en Postgres...
Ingesta exitosa: yellow 2025-03
[yellow 2025-03] Eliminado local: yellow_tripdata_2025-03.parquet


In [ ]:
# Comprobación de tablas RAW y conteos por partición (año/mes)
import os
import psycopg2
from typing import List, Tuple

# Reusar variables de entorno si ya existen, o tomarlas del entorno del contenedor
PG_HOST = os.getenv('PG_HOST')
PG_PORT = os.getenv('PG_PORT')
PG_DB = os.getenv('PG_DB')
PG_USER = os.getenv('PG_USER')
PG_PASSWORD = os.getenv('PG_PASSWORD')
PG_SCHEMA_RAW = os.getenv('PG_SCHEMA_RAW')

print(f"Usando Postgres: {PG_HOST}:{PG_PORT}/{PG_DB} schema={PG_SCHEMA_RAW}")

def connect():
    return psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASSWORD,
        connect_timeout=10,
    )

def table_exists(conn, schema: str, table: str) -> bool:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT EXISTS (
              SELECT 1
              FROM information_schema.tables
              WHERE table_schema = %s AND table_name = %s
            );
            """,
            (schema, table),
        )
        return cur.fetchone()[0]

def total_rows(conn, full_table: str) -> int:
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM {full_table};")
        return cur.fetchone()[0]

def counts_by_partition(conn, full_table: str) -> List[Tuple[int,int,int]]:
    with conn.cursor() as cur:
        cur.execute(
            f"""
            SELECT source_year, source_month, COUNT(*)
            FROM {full_table}
            GROUP BY 1,2
            ORDER BY 1,2
            """
        )
        return cur.fetchall()

ok_any = False
try:
    conn = connect()
    print("Conexión a Postgres OK\n")

    for service in ["yellow", "green"]:
        tbl = f"{PG_SCHEMA_RAW}.{service}_taxi_trip"
        exists = table_exists(conn, PG_SCHEMA_RAW, f"{service}_taxi_trip")
        if not exists:
            print(f"- {tbl}: NO EXISTE (aún no se ha cargado)")
            continue

        total = total_rows(conn, tbl)
        parts = counts_by_partition(conn, tbl)
        ok_any = ok_any or (total > 0)

        print(f"- {tbl}: filas={total:,} | particiones (año/mes)={len(parts)}")
        if parts:
            # últimas 3 particiones
            last3 = sorted(parts, key=lambda x: (x[0], x[1]))[-3:]
            print("  Últimas particiones cargadas:")
            for y, m, c in last3:
                print(f"    {y}-{m:02d}: {c:,} filas")
        print("")

    # Taxi zone lookup
    tz_exists = table_exists(conn, PG_SCHEMA_RAW, "taxi_zone_lookup")
    if tz_exists:
        tz_total = total_rows(conn, f"{PG_SCHEMA_RAW}.taxi_zone_lookup")
        print(f"- {PG_SCHEMA_RAW}.taxi_zone_lookup: filas={tz_total:,}")
    else:
        print(f"- {PG_SCHEMA_RAW}.taxi_zone_lookup: NO EXISTE (pendiente de cargar CSV)")

    conn.close()
except Exception as e:
    print("No se pudo conectar/consultar Postgres:", e)

print("\nResultado:")
if ok_any:
    print("✔ Verificación OK: Se encontraron filas en RAW (al menos en una tabla).")
else:
    print("✖ Verificación: No se encontraron filas en RAW. Revisa la ingesta.")

## Verificación de ingesta en Postgres

Esta sección valida que las tablas `raw.yellow_taxi_trip` y `raw.green_taxi_trip` existen y contienen datos. Además resume los conteos por año/mes y muestra las últimas particiones cargadas. También informa si `raw.taxi_zone_lookup` existe.

## Limpieza rápida de staging (opcional)
Ejecuta la siguiente celda para borrar el staging local del contenedor (`/home/jovyan/data/nyc_taxi_parquet`) cuando necesites liberar espacio. Esto no afecta los datos ya cargados en Postgres.

In [ ]:
# Borra staging del contenedor y reporta tamaño antes/después
import os, subprocess
stage = "/home/jovyan/data/nyc_taxi_parquet"
def du_h(path):
    try:
        out = subprocess.check_output(["bash","-lc",f"du -sh {path} || true"]).decode().strip()
        return out
    except Exception as e:
        return f"Error consultando tamaño: {e}"
print("Antes:", du_h(stage))
try:
    # remove contents only, keep folder
    for root, dirs, files in os.walk(stage):
        for f in files:
            try:
                os.remove(os.path.join(root, f))
            except Exception as e:
                print("No se pudo borrar:", f, e)
        for d in dirs:
            try:
                subprocess.call(["bash","-lc", f"rm -rf '{os.path.join(root, d)}'"])
            except Exception as e:
                print("No se pudo borrar dir:", d, e)
except Exception as e:
    print("Error limpiando staging:", e)
print("Después:", du_h(stage))

## Monitor de progreso de índices (opcional)
Este monitor imprime cada pocos segundos el estado de los `CREATE INDEX` activos y, si está disponible, el progreso detallado de `pg_stat_progress_create_index`.

Útil cuando no quieres cambiar a pgAdmin. Puedes ajustar el número de refrescos e intervalo con variables:
- `IDX_MONITOR_REFRESHES` (por defecto 12 ciclos)
- `IDX_MONITOR_INTERVAL` (por defecto 5 segundos por ciclo)

In [ ]:
# Ejecuta este monitor para ver progreso de índices en vivo
import os, time, psycopg2, textwrap

PG_HOST = os.getenv('PG_HOST')
PG_PORT = os.getenv('PG_PORT')
PG_DB = os.getenv('PG_DB')
PG_USER = os.getenv('PG_USER')
PG_PASSWORD = os.getenv('PG_PASSWORD')

REFRESHES = int(os.getenv('IDX_MONITOR_REFRESHES', '12'))
INTERVAL  = float(os.getenv('IDX_MONITOR_INTERVAL', '5'))


def _connect():
    return psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD, connect_timeout=5)


def show_activity():
    try:
        with _connect() as conn, conn.cursor() as cur:
            cur.execute(
                """
                SELECT pid,
                       now() - query_start AS runtime,
                       state,
                       LEFT(query, 160) AS query
                FROM pg_stat_activity
                WHERE datname = current_database()
                  AND query ILIKE 'create index%'
                  AND state <> 'idle'
                ORDER BY runtime DESC;
                """
            )
            rows = cur.fetchall()
            if not rows:
                print("(sin CREATE INDEX activos)")
                return
            print("CREATE INDEX activos:")
            for pid, runtime, state, q in rows:
                print(f"  pid={pid} | runtime={runtime} | state={state} | {q}")
    except Exception as e:
        print("[monitor] Error leyendo actividad:", e)


def show_progress():
    try:
        with _connect() as conn, conn.cursor() as cur:
            cur.execute(
                """
                SELECT pid,
                       relid::regclass::text AS table_name,
                       phase,
                       COALESCE(tuples_done,0)  AS tuples_done,
                       COALESCE(tuples_total,0) AS tuples_total,
                       COALESCE(blocks_done,0)  AS blocks_done,
                       COALESCE(blocks_total,0) AS blocks_total
                FROM pg_stat_progress_create_index
                ORDER BY pid;
                """
            )
            rows = cur.fetchall()
            if not rows:
                print("(sin filas en pg_stat_progress_create_index)")
                return
            print("Progreso detallado:")
            for pid, tbl, phase, td, tt, bd, bt in rows:
                pct_t = f"{(td/tt*100):.1f}%" if tt else "?"
                pct_b = f"{(bd/bt*100):.1f}%" if bt else "?"
                print(f"  pid={pid} | {tbl} | phase={phase} | tuples={td}/{tt} ({pct_t}) | blocks={bd}/{bt} ({pct_b})")
    except Exception as e:
        print("[monitor] Error leyendo progreso:", e)


print(f"Monitor de índices: {REFRESHES} ciclos, cada {INTERVAL}s\n")
for i in range(REFRESHES):
    print(f"--- Ciclo {i+1}/{REFRESHES} ---")
    show_activity()
    show_progress()
    if i < REFRESHES - 1:
        time.sleep(INTERVAL)
print("\nMonitor finalizado.")

## Ingesta GREEN robusta (manejo de esquema variable)

Los Parquet de Green (especialmente 2019) pueden traer columnas opcionales como `ehail_fee` que no siempre están en la tabla destino. Esta celda alinea el DataFrame al esquema real de `raw.green_taxi_trip` (añade columnas faltantes como NULL con el tipo correcto y descarta extras) antes de escribir por JDBC. Así evitamos errores y mantenemos la compatibilidad con índices y consultas.


In [8]:
# Green-only ingestion with schema alignment to DB table
import os
import psycopg2
from typing import List, Tuple
from pyspark.sql.functions import lit, col
from pyspark.sql.types import (
    StringType, IntegerType, LongType, DoubleType, BooleanType, TimestampType
)

# Reuse env from earlier cells
PG_HOST = os.getenv('PG_HOST')
PG_PORT = os.getenv('PG_PORT')
PG_DB = os.getenv('PG_DB')
PG_USER = os.getenv('PG_USER')
PG_PASSWORD = os.getenv('PG_PASSWORD')
PG_SCHEMA_RAW = os.getenv('PG_SCHEMA_RAW')

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/"
local_data_path = "/home/jovyan/data/nyc_taxi_parquet"
os.makedirs(local_data_path, exist_ok=True)

jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}?rewriteBatchedInserts=true"

def _connect():
    return psycopg2.connect(
        host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD, connect_timeout=10
    )

# Map Postgres data_type to Spark types (basic, sufficient for this dataset)
_pg_to_spark = {
    'integer': IntegerType(),
    'bigint': LongType(),
    'smallint': IntegerType(),
    'double precision': DoubleType(),
    'real': DoubleType(),
    'numeric': DoubleType(),
    'decimal': DoubleType(),
    'boolean': BooleanType(),
    'text': StringType(),
    'character varying': StringType(),
    'varchar': StringType(),
    'timestamp without time zone': TimestampType(),
    'timestamp with time zone': TimestampType(),
    'timestamp': TimestampType(),
}

def get_table_schema(schema: str, table: str) -> List[Tuple[str, str]]:
    """Return list of (column_name, data_type) in ordinal order for schema.table."""
    with _connect() as conn, conn.cursor() as cur:
        cur.execute(
            """
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = %s AND table_name = %s
            ORDER BY ordinal_position
            """,
            (schema, table),
        )
        return cur.fetchall()

def unify_df_to_table(df, table_cols: List[Tuple[str, str]]):
    """Rename df columns to lower-case, add missing columns with NULL cast to the DB type,
    drop extras, and reorder to DB column order."""
    # Lowercase all df columns to match unquoted PG names
    for c in df.columns:
        lc = c.lower()
        if lc != c:
            df = df.withColumnRenamed(c, lc)
    df_cols = set(df.columns)

    # Ensure existence and type for each DB column
    for col_name, pg_type in table_cols:
        if col_name not in df_cols:
            spark_type = _pg_to_spark.get(pg_type, StringType())
            df = df.withColumn(col_name, lit(None).cast(spark_type))
        # else: you could cast here if you want strict typing; usually not necessary
    # Reorder/select to DB order
    ordered_cols = [name for name, _ in table_cols]
    df = df.select(*ordered_cols)
    return df

# Controls like before
CHECK_EXISTS = os.getenv('CHECK_EXISTS', '1') == '1'
COUNT_ROWS  = os.getenv('COUNT_ROWS',  '0') == '1'

print(f"Ingesta GREEN con alineación de esquema. CHECK_EXISTS={int(CHECK_EXISTS)} COUNT_ROWS={int(COUNT_ROWS)}", flush=True)

# Pre-fetch DB schema for target table; if table doesn't exist aún, lo creará Spark en el primer append
try:
    target_cols = get_table_schema(PG_SCHEMA_RAW, 'green_taxi_trip')
    if not target_cols:
        print("Aviso: raw.green_taxi_trip no existe aún; se creará con el primer append.")
        target_cols = None
    else:
        print(f"Tabla destino encontrada con {len(target_cols)} columnas.")
except Exception as e:
    print("No se pudo leer el esquema de la tabla destino (se intentará crear automáticamente):", e)
    target_cols = None

for year in range(2021, 2026):
    for month in range(1, 13):
        service = 'green'
        file_name = f'{service}_tripdata_{year}-{month:02d}.parquet'
        url = f'{base_url}{file_name}'
        parquet_file = os.path.join(local_data_path, file_name)
        print(f"[{service} {year}-{month:02d}] Preparando ingesta...", flush=True)

        # Exists check by year/month
        if CHECK_EXISTS:
            try:
                with _connect() as conn, conn.cursor() as cur:
                    try:
                        cur.execute("SET LOCAL statement_timeout = '5s';")
                    except Exception:
                        pass
                    cur.execute(
                        f"""
                        SELECT EXISTS (
                          SELECT 1 FROM {PG_SCHEMA_RAW}.{service}_taxi_trip
                          WHERE source_year = %s AND source_month = %s
                        );
                        """,
                        (year, month),
                    )
                    if bool(cur.fetchone()[0]):
                        # remove local if any and skip
                        if os.path.exists(parquet_file):
                            try: os.remove(parquet_file)
                            except Exception: pass
                        print(f"[{service} {year}-{month:02d}] Ya existe en Postgres. Se omite.", flush=True)
                        continue
            except Exception as e:
                print(f"[{service} {year}-{month:02d}] Aviso: no se pudo consultar Postgres (se continúa): {e}", flush=True)

        # Download on demand
        if not os.path.exists(parquet_file):
            try:
                print(f"[{service} {year}-{month:02d}] Descargando {file_name} ...", flush=True)
                download_to(parquet_file, url)
                print(f"[{service} {year}-{month:02d}] Descargado: {file_name}", flush=True)
            except Exception as e:
                print(f"[{service} {year}-{month:02d}] Archivo no disponible {file_name}: {e}", flush=True)
                continue

        try:
            print(f"[{service} {year}-{month:02d}] Leyendo parquet...", flush=True)
            df = spark.read.parquet(parquet_file)
            # Add metadata columns before unification
            df = (
                df
                .withColumn('service_type', lit(service))
                .withColumn('source_year', lit(year))
                .withColumn('source_month', lit(month))
                .withColumn('ingested_at_utc', lit(datetime.utcnow()))
            )
            # Unify schema to destination table if known
            if target_cols is not None:
                df = unify_df_to_table(df, target_cols)
            else:
                # Lowercase column names for initial create
                for c in df.columns:
                    lc = c.lower()
                    if c != lc:
                        df = df.withColumnRenamed(c, lc)
            # Optional count
            if COUNT_ROWS:
                n_rows = df.count(); print(f"Filas leídas: {n_rows}", flush=True)
            else:
                n_rows = None; print("Filas leídas: N/A (COUNT_ROWS=0)", flush=True)

            print(f"[{service} {year}-{month:02d}] Escribiendo en Postgres...", flush=True)
            (
                df.write
                  .format('jdbc')
                  .option('url', jdbc_url)
                  .option('dbtable', f'{PG_SCHEMA_RAW}.{service}_taxi_trip')
                  .option('user', PG_USER)
                  .option('password', PG_PASSWORD)
                  .option('batchsize', '10000')
                  .mode('append')
                  .save()
            )
            if n_rows is not None:
                print(f"Ingesta exitosa: {service} {year}-{month:02d} | Filas: {n_rows}", flush=True)
            else:
                print(f"Ingesta exitosa: {service} {year}-{month:02d}", flush=True)
        except Exception as e:
            print(f"[{service} {year}-{month:02d}] Error en ingesta: {e}", flush=True)
        finally:
            if os.path.exists(parquet_file):
                try:
                    os.remove(parquet_file)
                    print(f"[{service} {year}-{month:02d}] Eliminado local: {file_name}", flush=True)
                except Exception as e:
                    print(f"[{service} {year}-{month:02d}] No se pudo eliminar {file_name}: {e}", flush=True)


Ingesta GREEN con alineación de esquema. CHECK_EXISTS=1 COUNT_ROWS=0
Tabla destino encontrada con 24 columnas.
[green 2021-01] Preparando ingesta...
[green 2021-01] Descargando green_tripdata_2021-01.parquet ...
[green 2021-01] Descargado: green_tripdata_2021-01.parquet
[green 2021-01] Leyendo parquet...
Filas leídas: N/A (COUNT_ROWS=0)
[green 2021-01] Escribiendo en Postgres...
Ingesta exitosa: green 2021-01
[green 2021-01] Eliminado local: green_tripdata_2021-01.parquet
[green 2021-02] Preparando ingesta...
[green 2021-02] Descargando green_tripdata_2021-02.parquet ...
[green 2021-02] Descargado: green_tripdata_2021-02.parquet
[green 2021-02] Leyendo parquet...
Filas leídas: N/A (COUNT_ROWS=0)
[green 2021-02] Escribiendo en Postgres...
Ingesta exitosa: green 2021-02
[green 2021-02] Eliminado local: green_tripdata_2021-02.parquet
[green 2021-03] Preparando ingesta...
[green 2021-03] Descargando green_tripdata_2021-03.parquet ...
[green 2021-03] Descargado: green_tripdata_2021-03.parqu

## Carga de Taxi Zone Lookup a raw.taxi_zone_lookup

Esta sección carga el archivo taxi_zone_lookup.csv (ya descargado) a Postgres en raw.taxi_zone_lookup, agregando el metadato ingested_at_utc. Es idempotente: trunca la tabla y reescribe todo el catálogo.


In [25]:
# Ingesta robusta taxi_zone_lookup.csv -> raw.taxi_zone_lookup (column-safe)
import os, re, psycopg2
from datetime import datetime
from pyspark.sql.functions import col, lit
from pyspark.sql.types import IntegerType, StringType

# Variables de entorno
PG_HOST = os.getenv('PG_HOST')
PG_PORT = os.getenv('PG_PORT')
PG_DB = os.getenv('PG_DB')
PG_USER = os.getenv('PG_USER')
PG_PASSWORD = os.getenv('PG_PASSWORD')
PG_SCHEMA_RAW = os.getenv('PG_SCHEMA_RAW')

zone_lookup_path = "/home/jovyan/data/nyc_taxi_parquet/taxi_zone_lookup.csv"
jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}?rewriteBatchedInserts=true"

# 1) Asegurar esquema/tabla y normalizar posibles nombres legacy en DB
try:
    conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD)
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute(f"CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA_RAW};")
        cur.execute(
            """
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = %s AND table_name = %s
            ORDER BY ordinal_position
            """,
            (PG_SCHEMA_RAW, "taxi_zone_lookup"),
        )
        cols = [r[0] for r in cur.fetchall()]
        if not cols:
            cur.execute(
                f"""
                CREATE TABLE IF NOT EXISTS {PG_SCHEMA_RAW}.taxi_zone_lookup (
                  location_id     INTEGER,
                  borough         TEXT,
                  zone            TEXT,
                  service_zone    TEXT,
                  ingested_at_utc TIMESTAMP
                );
                """
            )
            print("Tabla creada: raw.taxi_zone_lookup")
        else:
            # Renombres en la tabla si existen columnas legacy
            if "locationid" in cols and "location_id" not in cols:
                cur.execute(f'ALTER TABLE {PG_SCHEMA_RAW}.taxi_zone_lookup RENAME COLUMN locationid TO location_id;')
                print("DB fix: RENAME COLUMN locationid -> location_id")
            if "service zone" in cols and "service_zone" not in cols:
                cur.execute(f'ALTER TABLE {PG_SCHEMA_RAW}.taxi_zone_lookup RENAME COLUMN "service zone" TO service_zone;')
                print('DB fix: RENAME COLUMN "service zone" -> service_zone')
            if "servicezone" in cols and "service_zone" not in cols:
                cur.execute(f'ALTER TABLE {PG_SCHEMA_RAW}.taxi_zone_lookup RENAME COLUMN servicezone TO service_zone;')
                print("DB fix: RENAME COLUMN servicezone -> service_zone")
        cur.execute(f"TRUNCATE TABLE {PG_SCHEMA_RAW}.taxi_zone_lookup;")
        print("Tabla raw.taxi_zone_lookup lista (truncada).")
    conn.close()
except Exception as e:
    print("No se pudo preparar/normalizar la tabla raw.taxi_zone_lookup:", e)

# 2) Leer CSV con Spark
if not os.path.exists(zone_lookup_path):
    raise FileNotFoundError(f"No existe el archivo: {zone_lookup_path}")

print("Leyendo taxi_zone_lookup.csv ...")
df = spark.read.csv(zone_lookup_path, header=True, inferSchema=True)
print("Columnas originales:", df.columns)

# Quitar BOM si aparece en el primer encabezado
for c in df.columns:
    if isinstance(c, str) and c.startswith("\ufeff"):
        cleaned = c.lstrip("\ufeff")
        df = df.withColumnRenamed(c, cleaned)
        print(f"Se quitó BOM del encabezado: {c} -> {cleaned}")
        break

# 3) Detectar las columnas reales del CSV para cada destino
def find_src(df, target: str, candidates):
    # 1) Nombre exacto
    for cand in candidates:
        if cand in df.columns:
            return cand
    # 2) Por lower-case
    lowers = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lowers:
            return lowers[cand.lower()]
    # 3) Forma normalizada: lower + no alfanum -> _
    norm_map = {}
    for c in df.columns:
        norm = re.sub(r"[^0-9a-zA-Z]+", "_", c.strip()).strip("_").lower()
        norm_map.setdefault(norm, c)
    for cand in candidates:
        key = re.sub(r"[^0-9a-zA-Z]+", "_", cand.strip()).strip("_").lower()
        if key in norm_map:
            return norm_map[key]
    return None

src_map = {
    "location_id": find_src(df, "location_id", ["location_id","locationid","LocationID","location id","location-id"]),
    "borough":     find_src(df, "borough",     ["borough","Borough","borough "," BOROUGH"]),
    "zone":        find_src(df, "zone",        ["zone","Zone","zone "," ZONE"]),
    "service_zone":find_src(df, "service_zone",["service_zone","Service Zone","service zone","servicezone","service","SERVICE_ZONE"]),
}
print("Mapeo de columnas fuente -> destino:", src_map)

missing = [k for k,v in src_map.items() if v is None]
if missing:
    raise ValueError(f"No se localizaron columnas requeridas {missing} en el CSV. Columnas leídas: {df.columns}")

# 4) Reconstruir DataFrame limpio con select explícito + alias (usa backticks por si hay espacios o mayúsculas)
df_sel = (
    df.select(
        col(f"`{src_map['location_id']}`").cast(IntegerType()).alias("location_id"),
        col(f"`{src_map['borough']}`").cast(StringType()).alias("borough"),
        col(f"`{src_map['zone']}`").cast(StringType()).alias("zone"),
        col(f"`{src_map['service_zone']}`").cast(StringType()).alias("service_zone"),
    )
    .withColumn("ingested_at_utc", lit(datetime.utcnow()))
)

print("Esquema seleccionado:", df_sel.dtypes)
print("Escribiendo raw.taxi_zone_lookup ...")

# 5) Escribir a Postgres
(
    df_sel.write
      .format("jdbc")
      .option("url", jdbc_url)
      .option("dbtable", f"{PG_SCHEMA_RAW}.taxi_zone_lookup")
      .option("user", PG_USER)
      .option("password", PG_PASSWORD)
      .option("batchsize", "10000")
      .mode("append")
      .save()
)

# 6) Verificación rápida de filas
try:
    conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD)
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM {PG_SCHEMA_RAW}.taxi_zone_lookup;")
        n = cur.fetchone()[0]
        print(f"{PG_SCHEMA_RAW}.taxi_zone_lookup filas: {n}")
    conn.close()
except Exception as e:
    print("No se pudo verificar conteo de taxi_zone_lookup:", e)

DB fix: RENAME COLUMN locationid -> location_id
Tabla raw.taxi_zone_lookup lista (truncada).
Leyendo taxi_zone_lookup.csv ...
Columnas originales: ['LocationID', 'Borough', 'Zone', 'service_zone']
Mapeo de columnas fuente -> destino: {'location_id': 'LocationID', 'borough': 'Borough', 'zone': 'Zone', 'service_zone': 'service_zone'}
Esquema seleccionado: [('location_id', 'int'), ('borough', 'string'), ('zone', 'string'), ('service_zone', 'string'), ('ingested_at_utc', 'timestamp')]
Escribiendo raw.taxi_zone_lookup ...
raw.taxi_zone_lookup filas: 265
